# Lab 03 — Getting started with Hugging Face

Theo đề bài: suy luận cảm xúc bằng mô hình pretrained và fine-tune mô hình phân loại nhị phân bằng `Trainer`.

Notebook dùng chung mã nguồn với CLI. Chưa có kết quả full training được chèn sẵn: mọi metric, bảng và biểu đồ bên dưới được tạo khi bạn thực thi. Không dùng lại số liệu IMDB từ bài cũ.

**Chuẩn bị:** Python 3.12, cài `Lab3/requirements.txt`, chọn kernel tương ứng; bật Internet để tải Hugging Face. Bài 2 khuyến nghị Kaggle/Colab GPU. Xem README để cài PyTorch đúng CUDA. Không chạy pip tự động trong notebook để tránh thay đổi kernel ngoài ý muốn.

In [ ]:
from pathlib import Path
import sys
# Works from repository root, Lab3/, or Lab3/notebooks/.
LAB = next((p / "Lab3" if (p / "Lab3/src").is_dir() else p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "Lab3/src").is_dir() or (p / "src/workflow.py").is_file()), None)
if LAB is None:
    raise RuntimeError("Open this notebook inside a cloned Lab-Deep-Learning repository")
sys.path.insert(0, str(LAB))
import json
import torch
from IPython.display import display, Image, Markdown
from src.workflow import read_config, seed_all, exercise1, prepare_data, finetune
config = read_config()
seed_all(config["seed"])
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print(json.dumps(config, indent=2))

## Exercise 1 — Pretrained sentiment analysis

Sử dụng DistilBERT đã fine-tune trên SST-2. Tokenizer WordPiece biến văn bản thành ID; `[CLS]` mở đầu, `[SEP]` kết thúc; `attention_mask` đánh dấu token thật và padding. `pipeline` đóng gói tokenizer → model → softmax → nhãn.

Bước sau tải mô hình thật, chạy cả hai cách trên cùng câu và kiểm tra nhãn/xác suất khớp trong sai số `1e-5`.

In [ ]:
exercise1_run = exercise1(config)
print("Run artifacts:", exercise1_run)

In [ ]:
breakdown = json.loads((exercise1_run / "tokenizer_breakdown.json").read_text())
import pandas as pd
display(pd.DataFrame({k: breakdown[k] for k in ["tokens", "input_ids", "attention_mask"]}))
predictions = json.loads((exercise1_run / "predictions.json").read_text())
display(pd.DataFrame(predictions["manual"]))

### Tự kiểm tra
- ID và mask có cùng độ dài không? Tại sao `[CLS]` và `[SEP]` có mask bằng 1?
- Nhãn từ pipeline có khớp cách suy luận thủ công không?
- Thêm câu có phủ định hoặc cấu trúc nhượng bộ; quan sát độ tin cậy. Confidence không đảm bảo dự đoán đúng.

## Exercise 2 — Full Rotten Tomatoes dataset

Dùng **toàn bộ** `cornell-movie-review-data/rotten_tomatoes`: **8.530 train / 1.066 validation / 1.066 test**. Không lấy subset, không chia lại tập validation. Code từ chối dataset có kích thước khác hợp đồng này.

Train dùng để cập nhật trọng số; validation chọn checkpoint theo accuracy; test chỉ dùng đánh giá cuối. Không điều chỉnh siêu tham số dựa trên test.

In [ ]:
raw, tokenizer, tokenized = prepare_data(config)
display(raw)
print({k: len(raw[k]) for k in ["train", "validation", "test"]})
import pandas as pd
for split in ["train", "validation", "test"]:
    print(split, pd.Series(raw[split]["label"]).value_counts().sort_index().to_dict())

### Tokenization và dynamic padding
Cắt các chuỗi vượt quá 128 tokens; để `DataCollatorWithPadding` đệm đến chuỗi dài nhất của từng batch. Độ dài gốc và tỉ lệ truncation được đo thực tế trong mỗi run.

DistilBERT nền được tải mới với đầu phân loại 2 nhãn. Sanity overfit nằm trong tests trên mô hình dùng riêng, không làm thay đổi mô hình chính.

In [ ]:
print(tokenized["train"][0])
from transformers import DataCollatorWithPadding
batch = DataCollatorWithPadding(tokenizer)([tokenized["train"][i] for i in range(2)])
print({k: tuple(v.shape) for k, v in batch.items()})

### Trainer và fine-tuning
3 epochs, train/eval batch 16/32, AdamW, learning rate `2e-5`, warmup ratio `0.1`, weight decay `0.01`. Đánh giá và lưu mỗi epoch; khôi phục checkpoint tốt nhất theo validation accuracy. FP16 chỉ bật khi có CUDA.

Lệnh sau thực hiện huấn luyện đầy đủ và đánh giá độc lập 1.066 mẫu test. Máy không có GPU sẽ dừng với thông báo rõ ràng, tránh vô tình chạy job CPU dài.

In [ ]:
finetuning_run = finetune(config)
print("Run artifacts:", finetuning_run)

In [ ]:
metrics = json.loads((finetuning_run / "test_metrics.json").read_text())
display(pd.DataFrame([metrics]))
for filename in ["learning_curves.png", "confusion_matrix.png", "metrics_bar.png",
                 "token_length_distribution.png", "sample_predictions.png"]:
    print(filename)
    display(Image(filename=str(finetuning_run / filename)))

### Phân tích lỗi và tính tái lập
Xem các lỗi có confidence cao cùng những ví dụ đúng của cả hai lớp. Kiểm tra phủ định, nhượng bộ, mỉa mai và từ ngữ mô tả nội dung phim. Các giả thuyết ngôn ngữ không chứng minh nguyên nhân mô hình; bổ sung nhận xét thủ công sau khi đọc câu thực tế.

Mỗi run lưu config, phiên bản môi trường, model commit, dataset fingerprints, log, toàn bộ dự đoán, SHA256 manifest và checkpoint. Run mới không ghi đè run cũ.

In [ ]:
display(pd.DataFrame(json.loads((finetuning_run / "sample_predictions.json").read_text())))
display(Markdown((finetuning_run / "report.md").read_text()))
print("Reload check:", json.loads((finetuning_run / "reload_check.json").read_text()))